In [1]:
!pip install sentence-transformers faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import faiss
import time

from sentence_transformers import SentenceTransformer

import google.generativeai as genai

from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
GOOGLE_API_KEY = userdata.get(
    "Research++"
)

genai.configure(
    api_key=GOOGLE_API_KEY
)

llm = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print(
    "Gemini configured successfully."
)

Gemini configured successfully.


In [4]:
from google.colab import files

uploaded = files.upload()
uploaded = files.upload()

Saving papers.json to papers.json


Saving researchforge_faiss.index to researchforge_faiss.index


In [5]:
df = pd.read_json(
    "papers.json"
)

faiss_index = faiss.read_index(
    "researchforge_faiss.index"
)

print(
    f"Loaded {len(df)} papers."
)

Loaded 100 papers.


In [6]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print(
    "Embedding model ready."
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.


In [7]:
def retrieve_papers(
    user_query,
    top_k=10
):

    query_embedding = embedding_model.encode(
        [user_query]
    )

    distances, indices = faiss_index.search(
        np.array(
            query_embedding
        ).astype("float32"),
        k=top_k
    )

    retrieved_papers = []

    for idx in indices[0]:

        retrieved_papers.append({

            "title":
            df.iloc[idx]["title"],

            "abstract":
            df.iloc[idx]["abstract"]

        })

    return retrieved_papers

In [8]:
def generate_research_report(
    user_query,
    retrieved_papers
):

    evidence_context = ""

    for paper in retrieved_papers:

        evidence_context += f"""

Title:
{paper['title']}

Abstract:
{paper['abstract']}

"""

    prompt = f"""

You are ResearchForge++.

You are an AI research copilot.

Research Question:

{user_query}

Evidence:

{evidence_context}

Perform the following:

1. Compare major methods

2. Identify contradictions

3. Identify limitations

4. Discover research gaps

5. Generate research ideas

6. Conduct a short debate

7. Produce a final evidence-grounded conclusion

Use only the retrieved evidence.

Keep the report structured.

"""

    for attempt in range(3):

        try:

            response = llm.generate_content(
                prompt
            )

            return response.text

        except Exception:

            print(
                f"Retry {attempt+1}/3 ..."
            )

            time.sleep(60)

    return (
        "Report generation failed."
    )

In [9]:
query = """
Find future research opportunities
in multimodal deepfake detection.
"""

papers = retrieve_papers(
    query
)

report = generate_research_report(
    query,
    papers
)

print(
    report
)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 35156.08ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2129.16ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1597.21ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 8405.57ms


## Future Research Opportunities in Multimodal Deepfake Detection

This report analyzes the provided evidence to identify current methodologies, limitations, and future research opportunities in multimodal deepfake detection.

---

### 1. Comparison of Major Methods

The evidence reveals several approaches to multimodal deepfake detection:

*   **Large Multimodal Models (LMMs) with Supervised Fine-Tuning (SFT):** AV-LMMDetect leverages models like Qwen 2.5 Omni, casting detection as a prompted yes/no classification. It employs a two-stage training (LoRA alignment followed by audio-visual encoder fine-tuning) to jointly analyze audio and visual streams. These models aim to overcome the scaling and generalization issues of smaller, task-specific models.
*   **Integrated Fine-Grained Identification and Binary Classification:** One method proposes integrating fine-grained deepfake identification with binary classification, categorizing samples into four types by combining single-modality l